In [3]:
import os
import os.path
import sys
import numpy as np
import networkx as nx
import itertools as it
import random as rd
import math
import pickle as pk
import pandas as pd
from collections import defaultdict, Counter, ChainMap
from scipy.spatial import distance_matrix
import matplotlib.pyplot as plt
# %matplotlib inline

plt.rcParams['font.family'] = 'Arial'

## Visualizing ERK Baits and Prey networks

In [4]:
active_baits = [
    'BrafG469A', 'ArafS214P', 'HrasQ61R', 'HrasG13R', 
    'NrasQ61R', 'NrasQ61K', 
    'KrasG12V', 'KrasG12D',
    'ERK2DD', 'ERK1DD', 'MEK2DD', 'MEK1DD', 
    'Raf1S259F', 'Raf1S257L', 'BrafV600E']

inactive_baits = [
    'BrafD594N',  'ArafD447A', 
    'HrasS17N', 'NrasS17N', 'KrasS17N', 
    'ERK2AA', 'ERK1AA', 'MEK2AA', 'MEK1AA',
    'Raf1D486A']

wt_baits =  [ "BrafWT", "ArafWT", "HrasWT", "NrasWT", 
             "KrasWT","ERK2WT", "ERK1WT", "MEK2WT", 
             "MEK1WT", "Raf1WT"]

active_baits = active_baits[::-1]
inactive_baits = inactive_baits[::-1]
wt_baits = wt_baits[::-1]

active_baits = [x.upper() for x in active_baits]
inactive_baits = [x.upper() for x in inactive_baits]
wt_baits = [x.upper() for x in wt_baits]

In [5]:
import networkx as nx
import matplotlib.pyplot as plt
import random

# --- Groups ---
def build_graph(humap3_complex: pd.DataFrame, state: str):
    group1 = wt_baits
    
    if state == 'Active':
        group1 = active_baits
    elif state == 'WT':
        group1 = wt_baits
    else:
        group1 = inactive_baits
        
    print(state)
    # --- Build full graph ---
    G = nx.Graph()
    G.add_nodes_from(group1, bait="yes")

    group2 = set(humap3_complex['to'].values)
    G.add_nodes_from(group2, bait='no')

    df = humap3_complex[humap3_complex['from'].isin(group1)]
    for _, row in df.iterrows():
        G.add_edge(row["from"], row["to"], interaction=row['strict_type'])

    # Add G2–G2 edges with attributes (randomly assign cp or sg)
    df = humap3_complex[~humap3_complex['from'].isin(group1)]
    for _, row in df.iterrows():
        G.add_edge(row["from"], row["to"], interaction=row['strict_type'])

    # --- Layout ---
    pos_group1 = nx.circular_layout(group1, scale=3)
    pos = nx.spring_layout(G, pos=pos_group1, fixed=group1, k=0.7, scale=2, seed=14)
    return G, pos_group1, pos, group1, group2

def move2center(pos, group2):
    x = [] 
    y = []
    for elem in group2:
        a = pos[elem]
        x.append(a[0])
        y.append(a[1])

    for elem in group2:
        a = pos[elem]
        pos[elem] = [ a[0] - (np.mean(x) ), a[1] - (np.mean(y)) ]
    return pos


def draw_network(G, group1, group2, pos, ax, title=None):
    pos = move2center(pos, group2)
    plt.figure(figsize=(7, 7))
    # Draw nodes
    nx.draw_networkx_nodes(G, pos, nodelist=group1, node_color="#bdbdbd",
                           node_shape="D", node_size=400, edgecolors="none", ax=ax )
    nx.draw_networkx_nodes(G, pos, nodelist=group2, node_color="#bdbdbd",
                           node_shape="o", node_size=200, edgecolors="none", ax=ax)

    # Split edges by type
    edges_pp = [(u, v) for u, v, d in G.edges(data=True) if d["interaction"] == "both"]
    edges_apms = [(u, v) for u, v, d in G.edges(data=True) if d["interaction"] == "AP-MS"]
    edges_tid = [(u, v) for u, v, d in G.edges(data=True) if d["interaction"] == "TbID"]
    edges_cp = [(u, v) for u, v, d in G.edges(data=True) if d["interaction"] == "CORUM"]
    edges_sg = [(u, v) for u, v, d in G.edges(data=True) if d["interaction"] == "SIGNOR"]

    # Draw edges
    nx.draw_networkx_edges(G, pos, edgelist=edges_pp, edge_color="grey", width=1.5, alpha=0.8,ax=ax)
    nx.draw_networkx_edges(G, pos, edgelist=edges_apms, edge_color="#fb9a99", width=1.5, alpha=0.8,ax=ax)
    nx.draw_networkx_edges(G, pos, edgelist=edges_tid, edge_color="#a6cee3", width=1.5, alpha=0.8,ax=ax)
    nx.draw_networkx_edges(G, pos, edgelist=edges_cp, edge_color="black", width=2.5, alpha=0.8,ax=ax)
    nx.draw_networkx_edges(G, pos, edgelist=edges_sg, edge_color="#33a02c", width=2.5, alpha=0.8,ax=ax)

    label_pos = {node: (x, y - 0.3) for node, (x, y) in pos.items()}  # shift downward
    nx.draw_networkx_labels(G, label_pos, font_size=12, ax=ax)

def relabel_and_shift_graph(G, pos, prefix, shift=(0, 0), scale=55):
    """
    Relabel nodes with a prefix, scale + shift positions,
    return relabeled graph, position dict, and mapping.
    """
    mapping = {node: f"{prefix}_{node}" for node in G.nodes()}
    new_G = nx.relabel_nodes(G, mapping)
    
    # rescale + shift positions
    new_pos = {
        mapping[node]: (coord[0] * scale + shift[0], coord[1] * scale + shift[1])
        for node, coord in pos.items()
    }
    
    # keep original label as attribute
    nx.set_node_attributes(new_G, {mapping[node]: node for node in G.nodes()}, name="orig_label")
    
    return new_G, new_pos, mapping

**Combined perturbed - circular layout**



In [6]:
file_path = '/home/didusch92/Documents/dev/ErkPathwayReferenceMap/results/perturbations/'
file_name = 'merged_perturbations.tsv'
df_pert = pd.read_csv(file_path + file_name, sep='\t')

df_pert = df_pert[ df_pert['perturbed'] != 'unchanged' ]

df_pert['Mutant'] = df_pert['Mutant'].str.upper()

df_active = df_pert[df_pert["Mutant"].isin(active_baits) ]
print(df_active.head())


# --- Groups ---
def build_perturbation_graph(df_pert: pd.DataFrame, state: str):
    group1 = wt_baits
    
    if state == 'Active':
        group1 = active_baits
    elif state == 'WT':
        group1 = wt_baits
    else:
        group1 = inactive_baits
        
    print(state)
    # --- Build full graph ---
    G = nx.Graph()
    G.add_nodes_from(group1, bait="yes")

    group2 = set(df_pert['Gene.names'].values)
    G.add_nodes_from(group2, bait='no')

    for _, row in df_pert.iterrows():
        G.add_edge(row["Mutant"], row["Gene.names"], interaction=row['perturbed'])

    # --- Layout ---
    pos_group1 = nx.circular_layout(group1, scale=50)
    pos = nx.spring_layout(G, pos=pos_group1, fixed=group1, k=0.8, scale=5, seed=14)
    return G, pos_group1, pos, group1, group2


G_pert, pos_mutants, pos, mutants, diff_preys = build_perturbation_graph(df_active,  state='Active')

   Gene.names  logFC_APMS  padj_APMS  logFC_TbID  padj_TbID significant  \
2      ACTBL2    1.301142   0.002483    0.128144   0.559021       AP-MS   
4     ADAMTS1    1.094040   0.000559    0.472976   0.004531   congruent   
11     ANGPT2    1.944185   0.000852    2.061524   0.000581   congruent   
13    ANKRD22         NaN        NaN    2.189128   0.000477        TbID   
16   APOBEC3B         NaN        NaN    1.337784   0.000585        TbID   

       Mutant  combined_logfc  combined_padj perturbed  
2   BRAFG469A        1.301142       0.002483      gain  
4   BRAFG469A        1.094040       0.000559      gain  
11  BRAFG469A        2.061524       0.000581      gain  
13  BRAFG469A        2.189128       0.000477      gain  
16  BRAFG469A        1.337784       0.000585      gain  
Active


In [21]:
# --- Groups ---
def build_interactome_graph(df_pert: pd.DataFrame, state: str):
    group1 = wt_baits
    
    if state == 'Active':
        group1 = active_baits
    elif state == 'WT':
        group1 = wt_baits
    else:
        group1 = inactive_baits
        
    print(state)
    # --- Build full graph ---
    G = nx.Graph()
    G.add_nodes_from(group1, bait="yes")

    group2 = set(df_pert['prey'].values)
    G.add_nodes_from(group2, bait='no')

    for _, row in df_pert.iterrows():
        G.add_edge(row["bait"], 
                   row["prey"],
                   method=row['method'],
                   disgenet=row['HCDP'], #disgenet=row['DiseaseGene'],
                   leonetti=row['subcell_localization']) #leonetti=row['LeonettiLocalization'])

    # --- Layout ---
    pos_group1 = nx.circular_layout(group1, scale=50)
    pos = nx.spring_layout(G, pos=pos_group1, fixed=group1, k=0.8, scale=5, seed=14)
    return G, pos_group1, pos, group1, group2

In [22]:
file_path = '../../data/processed/erkpathmap_interactome.tsv'
interactome = pd.read_csv(file_path, sep='\t')

interactome_active = interactome[interactome["bait"].isin(active_baits) ]

G_active, pos_mutants, pos, mutants, diff_preys = build_interactome_graph(interactome_active,  state='Active')

Active


In [23]:
for node, (x, y) in pos_mutants.items():
    if node in G_active.nodes:
        G_active.nodes[node]['x'] = x * 10
        G_active.nodes[node]['y'] = y * 10
    else:
        print(node)

for node, (x, y) in pos.items():
    if node in G_active.nodes:
        G_active.nodes[node]['x'] = x * 10
        G_active.nodes[node]['y'] = y * 10
    else:
        print(node)

fname = "../../data/processed/cytoscape/graphml/combined_active_interactome.graphml"        
nx.write_graphml(G_active, 
                 fname)